# Qwen3-TTS 1.7B CustomVoice · Voice Studio v1.0.1

한국어 대사와 감정·말투 지시를 분리해 생성합니다. **이용자마다 본인 코랩에서 실행하세요.**

- **런타임 → 런타임 유형 변경 → T4 GPU**를 먼저 선택하세요.
- 공유 사이트에서 사용: **1번 설치 → 2번 모델 준비 → 4번 사이트 연결**.
- 무료 코랩 안에서 직접 사용: **1번 → 2번 → 3번**. 공유 사이트에서 목소리·스타일을 고른 뒤 **Qwen 코랩 직접 생성용 대본 받기**로 JSON 파일을 받으세요.
- 처음에는 모델 다운로드가 필요합니다. 준비 완료까지 기다린 뒤 다음 셀을 실행하세요.
- Cosy/GPT와 다른 전용 코랩에서 실행하세요. 재실행하면 준비된 Qwen 모델을 먼저 재사용합니다.
- Sohee는 한국어 여성 보이스입니다. 나머지 8개도 한국어를 읽을 수 있지만 모국어가 달라 발음·억양이 다를 수 있습니다.
- 모델 사용료는 없습니다. 무료 코랩 GPU 배정·시간은 제한되며, 외부 웹 화면 위주 사용은 제한 대상입니다. [Colab 공식 안내](https://research.google.com/colaboratory/faq.html)

[공유 사이트](https://voice-studio.streamlit.app/) · [Qwen 공식 설명](https://github.com/QwenLM/Qwen3-TTS)

코드 작성 후 실행 테스트나 샘플 음성 생성은 수행하지 않았습니다.

**v1.0.1:** 설치·준비 진행 로그를 셀에 직접 표시합니다. 4번 셀은 저장된 연결 주소를 다시 읽어 표시합니다. `returncode=0`만으로 완료를 안내하지 않습니다.


In [ ]:
#@title 1️⃣ Qwen 전용 설치 (처음 한 번)
from pathlib import Path
from runpy import run_path
qwen_root = Path('/content/voice_studio_qwen_v1')
qwen_root.mkdir(parents=True, exist_ok=True)
qwen_sources = {
    'qwen_colab_server.py': '"""Personal Colab runner for Qwen3-TTS 1.7B CustomVoice, release 1.0.0.\n\nThis module is copied into the notebook, so a downloaded notebook is complete.\nImporting this module never installs packages or starts inference.\n"""\nimport argparse\nfrom collections import deque\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport queue\nimport re\nimport secrets\nimport socket\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.request\nimport wave\n\nROOT = Path(\'/content/voice_studio_qwen_v1\')\nPYTHON = ROOT / \'venv/bin/python\'\nSTATE = ROOT / \'state.json\'\nMODEL_ID = \'Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\'\nSERVICE = \'voice-studio-qwen-customvoice\'\nVERSION = \'1.0.0\'\nVOICES = (\'Sohee\', \'Serena\', \'Vivian\', \'Ono_Anna\', \'Uncle_Fu\', \'Ryan\', \'Aiden\', \'Dylan\', \'Eric\')\n\n\ndef save_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False), encoding=\'utf-8\')\n    temporary.chmod(0o600)\n    temporary.replace(path)\n\n\ndef local_request(path, data=None):\n    state = json.loads(STATE.read_text())\n    url = f"http://127.0.0.1:{state[\'port\']}/v1/{state[\'token\']}" + path\n    request = urllib.request.Request(url, data=None if data is None else json.dumps(data).encode(),\n                                     headers={\'Content-Type\': \'application/json\'})\n    with urllib.request.urlopen(request, timeout=30) as response:\n        return json.load(response)\n\n\ndef setup():\n    ROOT.mkdir(parents=True, exist_ok=True, mode=0o700)\n    marker = ROOT / \'installed.json\'\n    if marker.is_file() and PYTHON.is_file() and json.loads(marker.read_text()).get(\'version\') == VERSION:\n        print(\'✅ Qwen 설치 파일이 준비되어 있습니다. 2번 셀을 실행하세요.\', flush=True)\n        return\n    import shutil\n    if not shutil.which(\'nvidia-smi\') or subprocess.run([\'nvidia-smi\'], capture_output=True).returncode != 0:\n        raise RuntimeError(\'런타임 → 런타임 유형 변경 → T4 GPU를 선택한 뒤 다시 실행하세요.\')\n    print(\'▶ 1/3 Qwen 전용 설치 공간 준비\', flush=True)\n    # No ensurepip/apt Python-version dependency; pip --python can bootstrap a venv.\n    subprocess.check_call([sys.executable, \'-m\', \'venv\', \'--without-pip\', str(ROOT / \'venv\')])\n    pip = [sys.executable, \'-m\', \'pip\', \'--python\', str(PYTHON)]\n    print(\'▶ 2/3 PyTorch 설치 · 다운로드 중에는 시간이 걸립니다.\', flush=True)\n    subprocess.check_call(pip + [\'install\', \'torch==2.8.0\', \'torchaudio==2.8.0\',\n                                \'--index-url\', \'https://download.pytorch.org/whl/cu126\'])\n    print(\'▶ 3/3 Qwen CustomVoice와 연결 도구 설치\', flush=True)\n    subprocess.check_call(pip + [\'install\', \'qwen-tts==0.1.1\', \'fastapi>=0.115,<1\',\n                                \'uvicorn>=0.30,<1\', \'soundfile>=0.12,<1\'])\n    # sox is an upstream runtime dependency, separate from its Python wrapper.\n    subprocess.check_call([\'apt-get\', \'-qq\', \'update\'])\n    subprocess.check_call([\'apt-get\', \'-qq\', \'install\', \'-y\', \'sox\', \'ffmpeg\'])\n    save_json(marker, {\'version\': VERSION})\n    print(\'✅ 설치 완료. 2번 셀에서 모델을 준비하세요.\', flush=True)\n\n\ndef start():\n    if not PYTHON.is_file() or not (ROOT / \'installed.json\').is_file():\n        raise RuntimeError(\'먼저 1번 설치 셀을 끝까지 실행해주세요.\')\n    state = {}\n    if STATE.is_file():\n        state = json.loads(STATE.read_text())\n        try:\n            status = local_request(\'/health\')\n            if status.get(\'service\') == SERVICE and status.get(\'version\') == VERSION and status.get(\'ready\'):\n                print(\'✅ 기존 Qwen 모델을 재사용합니다. 3번(직접 생성) 또는 4번(사이트 연결)으로 이동하세요.\')\n                return\n        except (OSError, ValueError):\n            pass\n        # An earlier cell may have been interrupted while its child still loads.\n        pid = state.get(\'pid\')\n        command = Path(f\'/proc/{pid}/cmdline\') if isinstance(pid, int) else None\n        if command and command.is_file() and b\'qwen_colab_server.py\\x00serve\' in command.read_bytes():\n            print(\'▶ 이미 시작한 Qwen 모델 준비를 이어서 기다립니다.\', flush=True)\n            wait_ready(pid)\n            return\n    with socket.socket() as sock:\n        sock.bind((\'127.0.0.1\', 0))\n        port = sock.getsockname()[1]\n    state = {\'port\': port, \'token\': secrets.token_urlsafe(32), \'version\': VERSION}\n    save_json(STATE, state)\n    environment = dict(os.environ, PYTHONUNBUFFERED=\'1\', TOKENIZERS_PARALLELISM=\'false\',\n                       HF_HOME=str(ROOT / \'hf_cache\'), PYTORCH_CUDA_ALLOC_CONF=\'expandable_segments:True\')\n    with (ROOT / \'server.log\').open(\'w\') as log:\n        process = subprocess.Popen([str(PYTHON), str(Path(__file__).resolve()), \'serve\'],\n                                   stdout=log, stderr=subprocess.STDOUT, env=environment, start_new_session=True)\n    state[\'pid\'] = process.pid\n    save_json(STATE, state)\n    wait_ready(process.pid)\n\n\ndef wait_ready(pid):\n    print(\'▶ Qwen 1.7B 모델을 내려받아 GPU에 올립니다. 첫 실행은 다운로드 때문에 오래 걸릴 수 있습니다.\', flush=True)\n    started, last_log = time.monotonic(), \'\'\n    while time.monotonic() - started < 1800:\n        try:\n            status = local_request(\'/health\')\n            if status.get(\'ready\') and status.get(\'service\') == SERVICE:\n                print(f"✅ 모델 준비 완료 · {status[\'gpu\']} · {status[\'precision\']} · GPU 배치 최대 {status[\'batch_size\']}개", flush=True)\n                print(\'공유 사이트는 4번 셀, 코랩 안에서 직접 생성하려면 3번 셀을 실행하세요.\', flush=True)\n                return\n        except (OSError, ValueError):\n            pass\n        log = (ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:]\n        if log != last_log:\n            print(log if not last_log else log[-1200:], flush=True)\n            last_log = log\n        status_path = Path(f\'/proc/{pid}/status\')\n        if not status_path.exists() or re.search(r\'^State:\\s+Z\', status_path.read_text(), re.M):\n            raise RuntimeError(\'Qwen 모델 준비가 중단되었습니다. 바로 위 오류를 확인해주세요.\')\n        time.sleep(3)\n    raise RuntimeError(\'모델 준비가 30분을 넘었습니다. server.log의 다운로드·GPU 오류를 확인해주세요. 자동으로 재시작하지 않았습니다.\')\n\n\ndef tunnel():\n    status = local_request(\'/health\')\n    if not status.get(\'ready\') or status.get(\'service\') != SERVICE:\n        raise RuntimeError(\'먼저 2번 셀의 모델 준비 완료를 기다려주세요.\')\n    state = json.loads(STATE.read_text())\n    previous = state.get(\'tunnel_pid\')\n    command = Path(f\'/proc/{previous}/cmdline\') if isinstance(previous, int) else None\n    if command and command.is_file() and b\'cloudflared\' in command.read_bytes() and state.get(\'public_url\'):\n        print(\'✅ 프로그램 연결 주소 (아래 전체를 Qwen 코랩 주소 칸에 넣으세요)\\n\' + state[\'public_url\'])\n        return\n    binary = ROOT / \'cloudflared\'\n    if not binary.is_file():\n        print(\'▶ Cloudflare 연결 도구 다운로드\', flush=True)\n        urllib.request.urlretrieve(\'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\', binary)\n        binary.chmod(0o700)\n    log_path = ROOT / \'tunnel.log\'\n    with log_path.open(\'w\') as log:\n        process = subprocess.Popen([str(binary), \'tunnel\', \'--url\', f"http://127.0.0.1:{state[\'port\']}",\n                                    \'--no-autoupdate\'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n    state[\'tunnel_pid\'] = process.pid\n    save_json(STATE, state)\n    for _ in range(60):\n        output = log_path.read_text(errors=\'replace\')\n        match = re.search(r\'https://[a-z0-9-]+\\.trycloudflare\\.com\', output)\n        if match:\n            state[\'public_url\'] = match.group() + \'/v1/\' + state[\'token\']\n            save_json(STATE, state)\n            print(\'✅ Qwen 공유 사이트 연결 준비 완료\\n프로그램 연결 주소:\\n\' + state[\'public_url\'], flush=True)\n            print(\'이 셀이 ▶로 돌아와도 서버는 실행 중입니다. 코랩 런타임은 유지해주세요.\')\n            return\n        if process.poll() is not None:\n            raise RuntimeError(\'공유 연결을 만들지 못했습니다. 3번 셀에서 직접 생성할 수 있습니다.\\n\' + output[-1000:])\n        time.sleep(1)\n    process.terminate()\n    raise RuntimeError(\'연결 주소를 받지 못했습니다. 4번 셀을 다시 실행하거나 3번 셀에서 직접 생성해주세요.\')\n\n\ndef split_text(text, maximum=240):\n    parts = []\n    while len(text) > maximum:\n        candidates = [m.end() for m in re.finditer(r\'[.!?。！？]\\s+|\\n+\', text[:maximum + 1]) if m.end() >= 40]\n        cut = candidates[-1] if candidates else text.rfind(\' \', 40, maximum + 1)\n        if cut < 40:\n            cut = maximum\n        parts.append(text[:cut].strip())\n        text = text[cut:].strip()\n    if text:\n        parts.append(text)\n    return parts\n\n\ndef join_wavs(paths, output, pause_ms=0):\n    parameters = None\n    with wave.open(str(output), \'wb\') as writer:\n        for number, path in enumerate(paths):\n            with wave.open(str(path), \'rb\') as reader:\n                current = (reader.getnchannels(), reader.getsampwidth(), reader.getframerate())\n                if parameters is None:\n                    parameters = current\n                    writer.setnchannels(current[0])\n                    writer.setsampwidth(current[1])\n                    writer.setframerate(current[2])\n                elif current != parameters:\n                    raise RuntimeError(\'음성 형식이 달라 원음 그대로 합칠 수 없습니다.\')\n                if number and pause_ms:\n                    writer.writeframes(b\'\\0\' * (current[2] * pause_ms // 1000) * current[0] * current[1])\n                while True:\n                    data = reader.readframes(65536)\n                    if not data:\n                        break\n                    writer.writeframes(data)\n\n\ndef serve():\n    import gc\n    import numpy as np\n    import soundfile as sf\n    import torch\n    from fastapi import FastAPI, HTTPException, Request\n    from fastapi.responses import FileResponse\n    import uvicorn\n    from qwen_tts import Qwen3TTSModel\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Qwen은 GPU 런타임을 사용합니다. CPU로 자동 전환하지 않았습니다.\')\n    # T4 lacks native BF16. Keep full precision there instead of forcing FP16.\n    dtype = (torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8\n             and torch.cuda.is_bf16_supported() else torch.float32)\n    print(f\'▶ {MODEL_ID} 불러오는 중 · {dtype} · SDPA\', flush=True)\n    model = Qwen3TTSModel.from_pretrained(MODEL_ID, device_map=\'cuda:0\', dtype=dtype,\n                                        attn_implementation=\'sdpa\')\n    model.model.eval()\n    print(\'✅ Qwen 모델을 GPU에 올렸습니다. 음성은 생성 버튼을 눌러야 만듭니다.\', flush=True)\n    state = json.loads(STATE.read_text())\n    audio_dir = ROOT / \'audio\'\n    audio_dir.mkdir(exist_ok=True, mode=0o700)\n    jobs, lock, pending = {}, threading.RLock(), queue.Queue()\n    capacity = {\'batch_size\': 2}\n    app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)\n    prefix = \'/v1/\' + state[\'token\']\n\n    def describe(job):\n        return {key: job[key] for key in (\'status\', \'chunks_done\', \'chunks\', \'synthesis_seconds\', \'batch_size\', \'error_code\')}\n\n    def find_job(job_id):\n        job = jobs.get(job_id)\n        if job is None:\n            raise HTTPException(404, \'작업 없음\')\n        return job\n\n    @app.get(prefix + \'/health\')\n    def health():\n        return dict(service=SERVICE, version=VERSION, protocol=1, model=MODEL_ID, ready=True,\n                    gpu=torch.cuda.get_device_name(0), precision=str(dtype), batch_size=capacity[\'batch_size\'])\n\n    @app.post(prefix + \'/jobs/{job_id}\')\n    async def submit(job_id: str, request: Request):\n        if not re.fullmatch(\'[a-f0-9]{32}\', job_id):\n            raise HTTPException(400, \'작업 번호 오류\')\n        body = bytearray()\n        async for chunk in request.stream():\n            body.extend(chunk)\n            if len(body) > 100000:\n                raise HTTPException(413, \'요청이 너무 큼\')\n        try:\n            data = json.loads(body)\n        except (ValueError, UnicodeError):\n            raise HTTPException(400, \'JSON 형식 오류\')\n        if not isinstance(data, dict):\n            raise HTTPException(400, \'요청 형식 오류\')\n        text, voice, instruct = data.get(\'text\'), data.get(\'voice\'), data.get(\'instruct\', \'\')\n        if (not isinstance(text, str) or not 1 <= len(text.strip()) <= 12000\n                or voice not in VOICES or not isinstance(instruct, str) or len(instruct) > 2500):\n            raise HTTPException(400, \'대사·목소리·스타일 오류\')\n        digest = hashlib.sha256(json.dumps([text, voice, instruct], ensure_ascii=False).encode()).hexdigest()\n        with lock:\n            if job_id in jobs:\n                job = jobs[job_id]\n                if digest != job[\'digest\']:\n                    raise HTTPException(409, \'작업 번호 중복\')\n                if job[\'status\'] not in (\'error\', \'cancelled\'):\n                    return describe(job)\n                del jobs[job_id]  # An explicit resubmission resumes a failed line.\n            if sum(job[\'status\'] in (\'queued\', \'generating\') for job in jobs.values()) >= 64 or len(jobs) >= 10000:\n                raise HTTPException(429, \'코랩 작업 대기열이 가득 참\')\n            texts = split_text(text.strip())\n            job = dict(id=job_id, digest=digest, texts=texts, voice=voice, instruct=instruct,\n                       status=\'queued\', chunks=len(texts), chunks_done=0, synthesis_seconds=0.0,\n                       batch_size=capacity[\'batch_size\'], error_code=\'\', paths=[])\n            jobs[job_id] = job\n            pending.put_nowait(job)\n            return describe(job)\n\n    @app.get(prefix + \'/jobs/{job_id}\')\n    def job_status(job_id: str):\n        with lock:\n            return describe(find_job(job_id))\n\n    @app.post(prefix + \'/jobs/{job_id}/cancel\')\n    def cancel_job(job_id: str):\n        with lock:\n            job = find_job(job_id)\n            # Finish and save an already-started line; cancel only untouched work.\n            if job[\'status\'] == \'queued\' and not job[\'chunks_done\']:\n                job[\'status\'] = \'cancelled\'\n            return describe(job)\n\n    @app.get(prefix + \'/jobs/{job_id}/audio\')\n    def audio(job_id: str):\n        with lock:\n            job = find_job(job_id)\n            if job[\'status\'] != \'done\':\n                raise HTTPException(409, \'아직 생성 중\')\n        return FileResponse(str(audio_dir / (job_id + \'.wav\')), media_type=\'audio/wav\')\n\n    def generate_batch(batch):\n        began = time.monotonic()\n        # Only this worker enters the model. Native batching avoids unsafe model\n        # calls from concurrent HTTP threads and repeated model loading.\n        with torch.inference_mode():\n            wavs, rate = model.generate_custom_voice(\n                text=[job[\'texts\'][job[\'chunks_done\']] for job in batch],\n                language=[\'Korean\'] * len(batch), speaker=[job[\'voice\'] for job in batch],\n                instruct=[job[\'instruct\'] for job in batch], non_streaming_mode=True)\n        if len(wavs) != len(batch):\n            raise RuntimeError(\'batch_count\')\n        elapsed = time.monotonic() - began\n        for job, wav in zip(batch, wavs):\n            if not len(wav) or not np.isfinite(wav).all():\n                with lock:\n                    job.update(status=\'error\', error_code=\'invalid_audio\')\n                continue\n            part = audio_dir / f"{job[\'id\']}_{job[\'chunks_done\']}.wav"\n            sf.write(str(part), wav, rate, subtype=\'PCM_16\')\n            with lock:\n                job[\'paths\'].append(part)\n                job[\'chunks_done\'] += 1\n                job[\'synthesis_seconds\'] += elapsed\n                job[\'batch_size\'] = len(batch)\n                if job[\'chunks_done\'] == job[\'chunks\']:\n                    output = audio_dir / (job[\'id\'] + \'.wav\')\n                    join_wavs(job[\'paths\'], output)\n                    job[\'status\'] = \'done\'\n                    for path in job[\'paths\']:\n                        path.unlink(missing_ok=True)\n                    job[\'paths\'] = []\n                    job[\'texts\'] = []\n                else:\n                    pending.put_nowait(job)\n\n    def process(batch):\n        memory_failure = False\n        try:\n            generate_batch(batch)\n        except torch.cuda.OutOfMemoryError:\n            memory_failure = True\n        except Exception:\n            import traceback\n            traceback.print_exc()\n            with lock:\n                for job in batch:\n                    if job[\'status\'] != \'done\':\n                        job.update(status=\'error\', error_code=\'generation\')\n        # Leave the exception handler before retrying, releasing its traceback\n        # and tensors so the smaller batch can actually reclaim GPU memory.\n        if memory_failure:\n            gc.collect()\n            torch.cuda.empty_cache()\n            if len(batch) > 1:\n                capacity[\'batch_size\'] = 1\n                print(\'GPU 메모리 부족: 모델·정밀도는 유지하고 배치 크기만 1개로 줄입니다.\', flush=True)\n                for job in batch:\n                    process([job])\n            else:\n                with lock:\n                    batch[0].update(status=\'error\', error_code=\'memory\')\n\n    def worker():\n        while True:\n            batch = [pending.get()]\n            deadline = time.monotonic() + 0.12\n            while len(batch) < capacity[\'batch_size\']:\n                try:\n                    batch.append(pending.get(timeout=max(0.001, deadline - time.monotonic())))\n                except queue.Empty:\n                    break\n                if time.monotonic() >= deadline:\n                    break\n            with lock:\n                batch = [job for job in batch if job[\'status\'] not in (\'cancelled\', \'done\', \'error\')]\n                for job in batch:\n                    job[\'status\'] = \'generating\'\n            if batch:\n                process(batch)\n\n    threading.Thread(target=worker, name=\'qwen-native-batch\', daemon=True).start()\n    uvicorn.run(app, host=\'127.0.0.1\', port=state[\'port\'], access_log=False, log_level=\'warning\')\n\n\ndef direct(plan_path):\n    """Generate inside a Colab cell, without any external web connection."""\n    plan = json.loads(Path(plan_path).read_text(encoding=\'utf-8-sig\'))\n    if plan.get(\'format\') != \'voice-studio-qwen-plan-v1\' or plan.get(\'model\') != MODEL_ID:\n        raise ValueError(\'공유 사이트에서 받은 Qwen 코랩 직접 생성용 JSON 파일을 올려주세요.\')\n    items = plan.get(\'items\', [])\n    if not items or len(items) > 5000:\n        raise ValueError(\'대사는 1~5,000개까지 넣어주세요.\')\n    indices = [item[\'index\'] for item in items]\n    if any(type(index) is not int or index <= 0 for index in indices) or len(set(indices)) != len(indices):\n        raise ValueError(\'대사 번호가 올바르지 않습니다.\')\n    items.sort(key=lambda item: item[\'index\'])\n    pause_ms = max(0, min(3000, int(plan.get(\'pause_ms\', 500))))\n    folder = ROOT / \'direct\' / hashlib.sha256(Path(plan_path).read_bytes()).hexdigest()[:16]\n    folder.mkdir(parents=True, exist_ok=True)\n    waiting, active, finished = deque(items), {}, {}\n    while waiting or active:\n        while waiting and len(active) < 4:\n            item = waiting.popleft()\n            output = folder / f"{item[\'index\']:04d}.wav"\n            if output.is_file():\n                finished[item[\'index\']] = output\n                continue\n            job_id = hashlib.sha256((str(folder) + str(item[\'index\'])).encode()).hexdigest()[:32]\n            payload = {name: item[name] for name in (\'text\', \'voice\', \'instruct\')}\n            local_request(\'/jobs/\' + job_id, payload)\n            active[job_id] = (item, output, time.monotonic())\n        for job_id, (item, output, submitted_at) in list(active.items()):\n            status = local_request(\'/jobs/\' + job_id)\n            if status[\'status\'] in (\'error\', \'cancelled\'):\n                raise RuntimeError(f"{item[\'index\']}번 생성 중단: {status.get(\'error_code\')}. 완료 음성은 {folder}에 보관했습니다.")\n            if time.monotonic() - submitted_at > 1800:\n                raise RuntimeError(f"{item[\'index\']}번이 30분 동안 완료되지 않았습니다. 5번 셀에서 로그를 확인해주세요. 완료 음성은 보관합니다.")\n            if status[\'status\'] == \'done\':\n                import shutil\n                shutil.copyfile(ROOT / \'audio\' / (job_id + \'.wav\'), output)\n                finished[item[\'index\']] = output\n                del active[job_id]\n                print(f"✅ {len(finished)}/{len(items)}개 저장 완료 · {item[\'index\']}번 {item.get(\'speaker\', \'\')}", flush=True)\n        if active:\n            time.sleep(1)\n    merged = folder / \'전체_음성.wav\'\n    join_wavs([finished[item[\'index\']] for item in items], merged, pause_ms)\n    mp3 = folder / \'전체_음성.mp3\'\n    subprocess.check_call([\'ffmpeg\', \'-nostdin\', \'-v\', \'error\', \'-y\', \'-i\', str(merged),\n                           \'-c:a\', \'libmp3lame\', \'-b:a\', \'192k\', str(mp3)])\n    def timestamp(seconds):\n        ms = round(seconds * 1000)\n        return f\'{ms // 3600000:02d}:{ms // 60000 % 60:02d}:{ms // 1000 % 60:02d},{ms % 1000:03d}\'\n    entries, cursor = [], 0.0\n    for number, item in enumerate(items, 1):\n        with wave.open(str(finished[item[\'index\']]), \'rb\') as reader:\n            end = cursor + reader.getnframes() / reader.getframerate()\n        text = (item.get(\'speaker\', \'\') + \': \' if plan.get(\'include_speaker\') else \'\') + item[\'text\']\n        entries.append(f\'{number}\\n{timestamp(cursor)} --> {timestamp(end)}\\n{text}\\n\')\n        cursor = end + pause_ms / 1000\n    (folder / \'자막.srt\').write_text(\'\\n\'.join(entries), encoding=\'utf-8-sig\')\n    import zipfile\n    bundle = folder / \'Qwen_전체음성_자막.zip\'\n    with zipfile.ZipFile(bundle, \'w\', zipfile.ZIP_STORED) as archive:\n        for path in (mp3, folder / \'자막.srt\'):\n            archive.write(path, path.name)\n    save_json(ROOT / \'direct_result.json\', {\'mp3\': str(mp3), \'wav\': str(merged), \'bundle\': str(bundle)})\n    print(\'✅ 전체 대사를 순서대로 합쳤습니다: \' + str(mp3), flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'action\', choices=[\'setup\', \'start\', \'serve\', \'tunnel\', \'direct\'])\n    parser.add_argument(\'--plan\')\n    args = parser.parse_args()\n    if args.action == \'direct\':\n        direct(args.plan)\n    else:\n        globals()[args.action]()\n',
    'qwen_colab_output.py': '"""Forward runner output to a Colab cell and show verified completion details.\n\nOnly standard-library imports; importing this file never starts a process.\nThe notebooks embed this helper so downloaded copies remain self-contained.\n"""\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport urllib.request\n\n\ndef _ready_state(runner):\n    expected_service = {\n        \'qwen_colab_server.py\': \'voice-studio-qwen-customvoice\',\n        \'qwen_voicebank_colab_server.py\': \'voice-studio-qwen-voicebank\',\n    }[runner.name]\n    try:\n        state = json.loads((runner.parent / \'state.json\').read_text(encoding=\'utf-8\'))\n        port, token = state[\'port\'], state[\'token\']\n        if not isinstance(port, int) or not 1 <= port <= 65535:\n            raise ValueError(\'Invalid local port\')\n        if not isinstance(token, str) or not re.fullmatch(r\'[A-Za-z0-9_-]{20,100}\', token):\n            raise ValueError(\'Invalid local token\')\n        request = urllib.request.Request(f\'http://127.0.0.1:{port}/v1/{token}/health\')\n        with urllib.request.urlopen(request, timeout=15) as response:\n            status = json.load(response)\n        if status.get(\'service\') != expected_service or not status.get(\'ready\'):\n            raise ValueError(\'Server is not ready\')\n        return state, status\n    except (OSError, ValueError, KeyError, TypeError):\n        raise RuntimeError(\'서버 준비 상태를 확인하지 못했습니다. 2번 셀을 실행하고 위에 표시되는 오류를 확인해주세요.\') from None\n\n\ndef run_qwen(runner, action, *arguments):\n    runner = Path(runner)\n    labels = {\n        \'setup\': \'1번 설치\', \'start\': \'2번 서버 준비\',\n        \'direct\': \'3번 전체 음성 생성\', \'tunnel\': \'4번 연결 주소 만들기\',\n        \'backup\': \'목소리 보관\', \'restore\': \'목소리 복원\',\n    }\n    if action not in labels:\n        raise ValueError(\'지원하지 않는 코랩 작업입니다.\')\n    if not runner.is_file():\n        raise RuntimeError(\'먼저 수정본의 1번 설치 셀을 실행해주세요.\')\n    print(f\'▶ {labels[action]} 시작 · 진행 내용이 아래에 표시됩니다.\', flush=True)\n    command = [sys.executable, \'-u\', str(runner), action, *map(str, arguments)]\n    # Notebook kernels do not always display inherited subprocess file descriptors.\n    # Read the child\'s output explicitly and print through the notebook\'s stream.\n    process = subprocess.Popen(\n        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n        text=True, encoding=\'utf-8\', errors=\'replace\', bufsize=1,\n        env=dict(os.environ, PYTHONUNBUFFERED=\'1\'),\n    )\n    try:\n        for line in process.stdout:\n            print(line, end=\'\', flush=True)\n        returncode = process.wait()\n    except KeyboardInterrupt:\n        process.terminate()\n        try:\n            process.wait(timeout=5)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n        print(\'\\n셀 대기를 중단했습니다. 다시 실행하면 서버 상태를 먼저 확인합니다.\', flush=True)\n        raise\n    finally:\n        process.stdout.close()\n    if returncode != 0:\n        raise RuntimeError(f\'{labels[action]} 중 오류가 발생했습니다. 바로 위 오류 내용을 확인해주세요. (종료 코드 {returncode})\')\n    # A zero exit code alone is not proof that the requested resource is ready.\n    if action == \'setup\':\n        if not (runner.parent / \'installed.json\').is_file() or not (runner.parent / \'venv/bin/python\').is_file():\n            raise RuntimeError(\'설치 완료 파일이 없습니다. 위 설치 로그를 확인하고 1번 셀을 다시 실행해주세요.\')\n        print(\'✅ 1번 설치 완료. 이제 2번 셀의 ▶를 눌러주세요.\', flush=True)\n    elif action == \'start\':\n        _, status = _ready_state(runner)\n        print(f"✅ 2번 서버 응답 확인 완료 · {status.get(\'gpu\', \'\')}", flush=True)\n        print(\'공유 사이트에서 사용하려면 3번을 건너뛰고 4번 셀의 ▶를 눌러주세요.\', flush=True)\n    elif action == \'tunnel\':\n        state, _ = _ready_state(runner)\n        url = state.get(\'public_url\', \'\')\n        if not isinstance(url, str) or not re.fullmatch(r\'https://[a-z0-9-]+\\.trycloudflare\\.com/v1/[A-Za-z0-9_-]{20,100}\', url) or not url.endswith(\'/v1/\' + state[\'token\']):\n            raise RuntimeError(\'연결 주소가 저장되지 않았습니다. 위 로그를 확인하고 4번 셀을 다시 실행해주세요.\')\n        label = (\'기본 목소리 20종 · 내 코랩 주소\'\n                 if runner.name == \'qwen_voicebank_colab_server.py\' else \'Qwen3-TTS · 내 코랩 주소\')\n        print(\'\\n✅ 프로그램 연결 주소 — 아래 한 줄 전체를 복사하세요.\', flush=True)\n        print(url, flush=True)\n        print(f\'공유 사이트 왼쪽 「{label}」 칸에 붙여넣고 연결 확인을 누르세요.\', flush=True)\n    # Return None so the notebook does not replace useful output with CompletedProcess.\n',
}
for qwen_name, qwen_source in qwen_sources.items():
    qwen_file = qwen_root / qwen_name
    qwen_file.write_text(qwen_source, encoding="utf-8")
    qwen_file.chmod(0o600)
qwen_runner = qwen_root / 'qwen_colab_server.py'
run_qwen = run_path(str(qwen_root / "qwen_colab_output.py"))["run_qwen"]
run_qwen(qwen_runner, "setup")


In [ ]:
#@title 2️⃣ Qwen 모델 준비 (준비 완료까지 기다리세요)
from pathlib import Path
from runpy import run_path
qwen_runner = Path('/content/voice_studio_qwen_v1/qwen_colab_server.py')
if not qwen_runner.is_file():
    raise RuntimeError('먼저 1번 설치 셀을 실행해주세요.')
qwen_output = qwen_runner.with_name('qwen_colab_output.py')
if not qwen_output.is_file():
    raise RuntimeError('수정본의 1번 설치 셀을 먼저 실행해주세요.')
run_qwen = run_path(str(qwen_output))['run_qwen']
run_qwen(qwen_runner, 'start')


## 3번은 코랩 안에서 직접 생성할 때 사용합니다
공유 사이트에서 사용할 경우 **3번을 건너뛰고 4번**으로 이동하세요.

직접 생성은 사이트에서 내려받은 `Qwen_대본_목소리_스타일.json`을 올리면 됩니다. 완료된 대사는 같은 런타임에서 재사용하고, 전부 완료되면 순서대로 MP3 하나로 합칩니다. 런타임이 삭제되면 저장한 결과도 사라지므로 다운로드하세요.


In [ ]:
#@title 3️⃣ 코랩에서 대본 전체 생성 → MP3 하나로 다운로드 (선택)
from google.colab import files
from IPython.display import Audio, display, FileLink
from pathlib import Path
from runpy import run_path
import json
qwen_root = Path('/content/voice_studio_qwen_v1')
qwen_runner = qwen_root / 'qwen_colab_server.py'
if not qwen_runner.is_file():
    raise RuntimeError('1번과 2번 셀을 먼저 실행해주세요.')
qwen_output = qwen_runner.with_name('qwen_colab_output.py')
if not qwen_output.is_file():
    raise RuntimeError('수정본의 1번 설치 셀을 먼저 실행해주세요.')
run_qwen = run_path(str(qwen_output))['run_qwen']
print('공유 사이트에서 받은 Qwen JSON 대본 파일 1개를 올려주세요.')
qwen_uploaded = files.upload()
qwen_json_files = [name for name in qwen_uploaded if name.lower().endswith('.json')]
if len(qwen_json_files) != 1:
    raise ValueError('Qwen JSON 파일 1개만 선택해주세요.')
qwen_plan_path = qwen_root / 'direct_input.json'
qwen_plan_path.write_bytes(qwen_uploaded[qwen_json_files[0]])
run_qwen(qwen_runner, 'direct', '--plan', qwen_plan_path)
qwen_result = json.loads((qwen_root / 'direct_result.json').read_text())
print('✅ 전체 MP3 생성 완료. 다운로드를 시작합니다.')
display(Audio(filename=qwen_result['mp3']))
files.download(qwen_result['mp3'])
print('MP3와 자막을 함께 보관하려면 아래 ZIP 파일도 받을 수 있습니다.')
display(FileLink(qwen_result['bundle']))


## 4번은 공유 사이트에서 생성할 때 사용합니다
무료 코랩은 외부 웹 화면을 주로 사용하는 작업이 종료될 수 있습니다. 무료로 코랩 안에서 직접 사용하려면 위 3번을 이용하세요.

아래에 나오는 **프로그램 연결 주소 전체**를 공유 사이트 왼쪽 **Qwen3-TTS · 내 코랩 주소**에 넣고 **Qwen 연결 확인**을 누르세요. 다른 이용자에게 주소를 보내지 말고, 각자 본인 코랩을 실행하세요.


In [ ]:
#@title 4️⃣ 공유 사이트 연결 주소 만들기 (선택)
from pathlib import Path
from runpy import run_path
qwen_runner = Path('/content/voice_studio_qwen_v1/qwen_colab_server.py')
if not qwen_runner.is_file():
    raise RuntimeError('먼저 1번과 2번 셀을 실행해주세요.')
qwen_output = qwen_runner.with_name('qwen_colab_output.py')
if not qwen_output.is_file():
    raise RuntimeError('수정본의 1번 설치 셀을 먼저 실행해주세요.')
run_qwen = run_path(str(qwen_output))['run_qwen']
run_qwen(qwen_runner, 'tunnel')


In [ ]:
#@title 5️⃣ 오류 로그 보기 (오류가 났을 때만)
from pathlib import Path
qwen_log = Path('/content/voice_studio_qwen_v1/server.log')
print(qwen_log.read_text(errors='replace')[-7000:] if qwen_log.is_file() else '아직 모델 준비 로그가 없습니다. 1번과 2번 셀부터 확인해주세요.')
